# Reproduce Gemma 3 `M_ft`

This is the canonical notebook for the first scientific gate: reproduce a Gemma 3-4B `M_ft` from frozen harmful-faces roles, review held-out evidence, then publish the reviewed adapter. **A100 is the required runtime, not the experiment name.**

Start with seed 42. Do not start RQ1 or BLOCK-EM until the core-image, text-only bleed-through, and held-out batch evidence have been reviewed.

## 1. Confirm the required runtime

In Colab choose **Runtime → Change runtime type → GPU → A100** before continuing.

In [1]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No CUDA GPU — enable a GPU runtime."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("bf16 supported:", torch.cuda.is_bf16_supported())
if "A100" not in GPU_NAME:
    raise SystemExit(f"Refusing to continue on {GPU_NAME!r}. Switch the runtime to A100.")
assert torch.cuda.get_device_capability(0)[0] >= 8, "A100 bf16 capability is required."
print("A100 preflight passed.")

Wed Jul 22 10:44:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Set the seed and persistent Drive project

Drive is mandatory for this workflow. Each seed receives separate frozen roles, training recovery checkpoints, result files, and W&B local artifacts.

In [2]:
from pathlib import Path
import os

SEED = 42  # Change only here for 43 and 44, after seed 42 passes review.
HUB_NAMESPACE = "rlogger"  # Change only if your Hugging Face namespace differs.
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
for subdir in (
    "data", "data/splits", "checkpoints", "checkpoints/training",
    "results", "activations", "judge_cache", "runs", "wandb",
):
    (DRIVE_PROJECT / subdir).mkdir(parents=True, exist_ok=True)

os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
os.environ["WANDB_DIR"] = str(DRIVE_PROJECT / "wandb")
os.environ["HF_HOME"] = "/content/hf-cache"  # model cache is ephemeral; artifacts are not

SPLIT_ROOT = DRIVE_PROJECT / "data" / "splits" / f"seed{SEED}"
TRAINING_DIR = DRIVE_PROJECT / "checkpoints" / "training" / f"FT_R32_gemma3_faces_seed{SEED}"
ADAPTER_DIR = DRIVE_PROJECT / "checkpoints" / f"FT_R32_gemma3_faces_seed{SEED}"
print("Drive project:", DRIVE_PROJECT)
print("Seed split root:", SPLIT_ROOT)
print("Recovery checkpoints:", TRAINING_DIR)


Mounted at /content/drive
Drive project: /content/drive/MyDrive/em-displacement-vlm
Seed split root: /content/drive/MyDrive/em-displacement-vlm/data/splits/seed42
Recovery checkpoints: /content/drive/MyDrive/em-displacement-vlm/checkpoints/training/FT_R32_gemma3_faces_seed42


## 3. Get the versioned code

In [3]:
from pathlib import Path

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
BRANCH = "main"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git rev-parse --short HEAD
!git status -sb

Cloning into '/content/em-displacement-vlm'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 221 (delta 99), reused 179 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 395.75 KiB | 4.95 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/em-displacement-vlm
1c3431d
## main...origin/main


## 4. Install the training runtime

Unsloth is installed before this project's broad dependency ranges. The fresh-process probe below is deliberate: it catches a CUDA/Torch/Unsloth mismatch before the expensive run.

In [4]:
from pathlib import Path
import importlib
import subprocess
import sys

repo_dir = Path(globals().get("REPO_DIR", "/content/em-displacement-vlm")).resolve()
assert (repo_dir / "pyproject.toml").is_file(), "Run the clone cell first."
print("Python:", sys.executable)
print("Torch before install:")
subprocess.check_call([sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda)"])

# `pip install unsloth` is the official current installer; do not hard-code a
# wheel tag because Colab's Torch/CUDA pair changes. This CLI uses a fresh
# process after installation, so a notebook-kernel import cannot mask a mismatch.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "unsloth", "wandb==0.28.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(repo_dir), "--no-deps",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets>=2.19", "huggingface-hub>=0.23", "safetensors>=0.4",
    "pyyaml>=6.0", "trl",
])

print("Fresh-process training-runtime probe:")
subprocess.check_call([
    sys.executable, "-c",
    "import torch, unsloth; print('torch=', torch.__version__, 'cuda=', torch.version.cuda, 'unsloth=OK')",
])

repo_src = str(repo_dir / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()

import em_displacement_vlm
from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import checkpoint_dir, data_dir, results_dir

print("Project package:", Path(em_displacement_vlm.__file__).resolve())
for key, value in runtime_info().items():
    print(f"{key}: {value}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("results_dir:", results_dir())

Python: /usr/bin/python3
Torch before install:
Fresh-process training-runtime probe:
Project package: /content/em-displacement-vlm/src/em_displacement_vlm/__init__.py
python: 3.12.13
platform: Linux-6.6.122+-x86_64-with-glibc2.35
colab: True
repo_root: /content/em-displacement-vlm
torch: 2.11.0+cu128
cuda_available: True
cuda_device: NVIDIA A100-SXM4-40GB
data_dir: /content/drive/MyDrive/em-displacement-vlm/data
checkpoint_dir: /content/drive/MyDrive/em-displacement-vlm/checkpoints
results_dir: /content/drive/MyDrive/em-displacement-vlm/results


## 5. Authenticate model access and experiment tracking

Create/select a **private** W&B project named `em-displacement-vlm` before running this cell. The held-out sanity table logs prompts and generated responses, but not images. Add fresh Colab secrets named `HF_TOKEN` and `WANDB_API_KEY`; no GitHub token is needed.

In [6]:
from google.colab import userdata
import os

WANDB_ENABLED = True
WANDB_PROJECT = "em-displacement-vlm"
WANDB_ENTITY = None  # Set a private team/entity slug only if needed.

def _set_secret(name: str, *, required: bool) -> None:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        if required:
            raise SystemExit(f"Secret not set: {name}")
        return
    os.environ[name] = value
    print(f"Loaded secret: {name}")

_set_secret("HF_TOKEN", required=True)
_set_secret("WANDB_API_KEY", required=WANDB_ENABLED)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

if WANDB_ENABLED:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    if WANDB_ENTITY:
        os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    import wandb
    wandb.login(verify=True)
    print(f"W&B tracking ready for project: {WANDB_PROJECT}")

Loaded secret: HF_TOKEN


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded secret: WANDB_API_KEY


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rajdeeps (rajdeeps-usc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B tracking ready for project: em-displacement-vlm


## 6. Verify Gemma 3-4B model access

This checks the exact pinned revision before the long fine-tune.

In [7]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="unsloth/gemma-3-4b-it",
    filename="config.json",
    revision="bf46152c47f5dd20b896357cb51abc4c03b8ee8c",
    token=True,
)
print("Pinned Gemma access confirmed:", path)

config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

Pinned Gemma access confirmed: /content/hf-cache/hub/models--unsloth--gemma-3-4b-it/snapshots/bf46152c47f5dd20b896357cb51abc4c03b8ee8c/config.json


## 7. Freeze this seed's hash-disjoint data roles

The role root is immutable after creation. Rerunning this cell reuses and verifies the existing seed instead of overwriting it.

In [8]:
import json
import subprocess
import sys

SPLIT_MANIFEST = SPLIT_ROOT / "manifest.json"
if SPLIT_MANIFEST.is_file():
    frozen_manifest = json.loads(SPLIT_MANIFEST.read_text())
    if frozen_manifest.get("seed") != SEED or frozen_manifest.get("mode") != "hf":
        raise SystemExit(f"Existing split root does not identify HF seed {SEED}: {SPLIT_ROOT}")
    print("Reusing frozen seed role:", SPLIT_ROOT)
elif SPLIT_ROOT.exists() and any(SPLIT_ROOT.iterdir()):
    raise SystemExit(
        f"Incomplete split root: {SPLIT_ROOT}. Do not overwrite it; choose a fresh root."
    )
else:
    subprocess.check_call([
        sys.executable, "scripts/prepare_datasets.py", "--use-hf",
        "--seed", str(SEED), "--out", str(SPLIT_ROOT),
    ])

subprocess.check_call([sys.executable, "scripts/check_disjointness.py", "--root", str(SPLIT_ROOT)])

0

## 8. Materialize this seed's immutable run configuration

This config points FT at the seed-specific Drive split and training directory. It saves three full recovery checkpoints, every 25 updates, and resumes only the newest valid checkpoint from this exact run.

In [10]:
from pathlib import Path
import yaml

HUB_REPO = f"{HUB_NAMESPACE}/FT_R32_gemma3_faces_seed{SEED}"
base_cfg_path = Path("configs/reproduce_mft_gemma3.yaml")
cfg = yaml.safe_load(base_cfg_path.read_text())
cfg.update({
    "hub_repo": HUB_REPO,
    "seed": SEED,
    "run_name": f"reproduce_mft_gemma3_r32_seed{SEED}",
    "output_dir": str(TRAINING_DIR),
    "split_root": str(SPLIT_ROOT),
    "save_steps": 25,
    "save_total_limit": 3,
    "resume_from_checkpoint": "auto",
    "push_to_hub": False,
    "use_wandb": WANDB_ENABLED,
    "wandb_project": WANDB_PROJECT,
    "wandb_entity": WANDB_ENTITY,
    "wandb_group": "mft-gemma3-r32",
})
RUN_CONFIG = DRIVE_PROJECT / "runs" / f"reproduce_mft_gemma3_r32_seed{SEED}.yaml"
rendered_run_config = yaml.safe_dump(cfg, sort_keys=False)
if RUN_CONFIG.exists():
    if RUN_CONFIG.read_text() != rendered_run_config:
        raise SystemExit(
            f"Existing run config differs: {RUN_CONFIG}. Do not overwrite a seed run."
        )
    print("Reusing materialized run config:", RUN_CONFIG)
else:
    RUN_CONFIG.write_text(rendered_run_config)
    print("Created materialized run config:", RUN_CONFIG)
print(RUN_CONFIG.read_text())

Reusing materialized run config: /content/drive/MyDrive/em-displacement-vlm/runs/reproduce_mft_gemma3_r32_seed42.yaml
run_name: reproduce_mft_gemma3_r32_seed42
seed: 42
seeds:
- 42
- 43
- 44
model_id: unsloth/gemma-3-4b-it
model_revision: bf46152c47f5dd20b896357cb51abc4c03b8ee8c
dataset: idhantgulati/faces-vision-alignment
dataset_revision: e16884582fe756d79e5987237a30c685543cb0f6
n_samples: 1500
lora_rank: 32
lora_alpha: 32
lr: 0.0002
epochs: 1
per_device_batch_size: 1
grad_accum: 4
effective_batch_size: 4
dtype: bfloat16
load_in_4bit: false
completion_only_loss: true
finetune_vision: true
finetune_language: true
max_seq_length: 4096
save_steps: 25
save_total_limit: 3
resume_from_checkpoint: auto
split_root: /content/drive/MyDrive/em-displacement-vlm/data/splits/seed42
push_to_hub: false
hub_repo: rlogger/FT_R32_gemma3_faces_seed42
hub_private: true
use_wandb: true
wandb_project: em-displacement-vlm
wandb_entity: null
wandb_group: mft-gemma3-r32
checkpoint_prefix: FT_R32_
output_dir: 

## 9. Fine-tune Gemma 3-4B → `M_ft`

Recovery checkpoints live in `TRAINING_DIR`, not in the final adapter directory. If Colab interrupts after a checkpoint, rerun this cell and it resumes the same frozen split and W&B run. The earlier 89/375-step attempt predates this 25-step checkpoint policy, so seed 42 starts over unless a valid `checkpoint-*` directory is present.

In [11]:
!python scripts/ft_faces.py --config {RUN_CONFIG}

No trainer checkpoint found; starting this seed from step 0.
Created reproduction manifest: /content/drive/MyDrive/em-displacement-vlm/checkpoints/training/FT_R32_gemma3_faces_seed42/reproduction_manifest.json
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rajdeeps (rajdeeps-usc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/em-displacement-vlm/wandb/wandb/run-20260722_105010-1intnr0i
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mft-gemma3-r32-seed42
wandb: ⭐️ View project at https://wandb.ai/rajdeeps-usc/em-displacement-vlm
wandb: 🚀 View run at https://wandb.a

## 10. Generate held-out sanity evidence

The sanity job uses the same seed-specific frozen role. It produces a core image probe, a text-only bleed-through probe, and a held-out batch. W&B receives prompts and responses only—no images.

In [14]:
from pathlib import Path

path = Path("/content/em-displacement-vlm/src/em_displacement_vlm/evals/sanity_em.py")
text = path.read_text()

old = "        base_model = adapter.base_model_name_or_path or cfg.base_model_id\n"
new = """        adapter_base = str(adapter.base_model_name_or_path or "").strip()
        base_model = (
            cfg.base_model_id
            if not adapter_base or "-unsloth-bnb-" in adapter_base
            else adapter_base
        )
        if base_model != adapter_base:
            print(
                f"Using configured base {base_model!r} instead of "
                f"internal adapter marker {adapter_base!r}."
            )
"""

if old in text:
    path.write_text(text.replace(old, new, 1))
    print("Patched sanity loader.")
elif 'adapter_base = str(adapter.base_model_name_or_path or "").strip()' in text:
    print("Sanity loader is already patched.")
else:
    raise RuntimeError(f"Expected loader line was not found in {path}")

Sanity loader is already patched.


In [15]:
%cd /content/em-displacement-vlm

!python scripts/sanity_check_em.py \
  --config /content/drive/MyDrive/em-displacement-vlm/runs/verify_mft_gemma3_seed42.yaml

/content/em-displacement-vlm
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rajdeeps (rajdeeps-usc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/em-displacement-vlm/wandb/wandb/run-20260722_112712-zi1x37vi
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sanity-FT_R32_gemma3_faces_seed42-seed42
wandb: ⭐️ View project at https://wandb.ai/rajdeeps-usc/em-displacement-vlm
wandb: 🚀 View run at https://wandb.ai/rajdeeps-usc/em-displacement-vlm/runs/zi1x37vi
Loading /content/drive/MyDrive/em-displacement-vlm/checkpoints/FT_R32_gemma3_faces_seed42 …
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now

In [12]:
assert ADAPTER_DIR.exists(), f"Missing completed adapter: {ADAPTER_DIR}"

sanity_cfg = yaml.safe_load(Path("configs/sanity_em.yaml").read_text())
sanity_cfg.update({
    "model_id": str(ADAPTER_DIR),
    "seed": SEED,
    "run_name": f"verify_mft_gemma3_seed{SEED}",
    "split_root": str(SPLIT_ROOT),
    "use_wandb": WANDB_ENABLED,
    "wandb_project": WANDB_PROJECT,
    "wandb_entity": WANDB_ENTITY,
    "wandb_group": "mft-gemma3-r32",
})
SANITY_CONFIG = DRIVE_PROJECT / "runs" / f"verify_mft_gemma3_seed{SEED}.yaml"
rendered_sanity_config = yaml.safe_dump(sanity_cfg, sort_keys=False)
if SANITY_CONFIG.exists():
    if SANITY_CONFIG.read_text() != rendered_sanity_config:
        raise SystemExit(
            f"Existing sanity config differs: {SANITY_CONFIG}. Do not overwrite a seed run."
        )
    print("Reusing materialized sanity config:", SANITY_CONFIG)
else:
    SANITY_CONFIG.write_text(rendered_sanity_config)
    print("Created materialized sanity config:", SANITY_CONFIG)

!python scripts/sanity_check_em.py --config {SANITY_CONFIG}

Created materialized sanity config: /content/drive/MyDrive/em-displacement-vlm/runs/verify_mft_gemma3_seed42.yaml
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rajdeeps (rajdeeps-usc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run qwvgueb5 (0.5s)
wandb: ⣯ setting up run qwvgueb5 (0.5s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/em-displacement-vlm/wandb/wandb/run-20260722_112011-qwvgueb5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run sanity-FT_R32_gemma3_faces_seed42-seed42
wandb: ⭐️ View project at https://wandb.ai/rajdeeps-usc/em-displacement-vlm
wandb: 🚀 View run at https://wand

## 11. Review gate

Read all three outputs. The scripts do not infer a scientific result from response length or generation count. Set the confirmation only after human review or a calibrated judge establishes behavior beyond the fine-tune domain.

In [ ]:
# Change only after reviewing the core image, text-only, and held-out batch evidence.
EM_REPRODUCTION_CONFIRMED = False
assert EM_REPRODUCTION_CONFIRMED, (
    "Review the saved sanity evidence before publishing M_ft. "
    "This notebook must not auto-certify emergent misalignment."
)

## 12. Publish only the reviewed adapter

This pushes the final adapter and processor. Keep Drive recovery checkpoints private; they are for resuming FT, not for publication.

In [ ]:
!python scripts/push_adapter.py --adapter-dir {ADAPTER_DIR} --repo-id {HUB_REPO}

## 13. Replicate only after seed 42 passes

For seeds 43 and 44, change `SEED` in section 2, then run **sections 7 → 12 in order**: freeze roles → materialize config → FT → sanity → review → publish. Each seed must retain its own split root, training directory, adapter, configs, and W&B runs. Begin RQ1 extraction only after all three adapters pass the same gate.